# Benchmark
A comprehensive comparison against state-of-the-art causal discovery methodologies is performed. <br>
Each selected method exemplifies a distinct category within the spectrum of causal inference approaches. <br>
Namely, we includes the Pairwise Granger Causality test implementation available in the `statsmodels` package for Python <br>

From the constraint-based family, we select the PCMCI algorithm developed by Runge, which is implemented in the Tigramite Python package. <br> 
The VarLiNGAM method, as proposed by Hyvärinen et al., is our choice for the noise-based category.  <br> 
It is implemented in Python via the LiNGAM library .   <br> 

Finally, for the score-based category, we incorporate DYNOTEARS, a method introduced by Pamfil et al. and implemented in the CausalNex Python library.  <br> 

We also include standard VAR modeling, adopting the regression coefficients as if they were causal.  <br> 


It is important to note that each method has its own underlying assumptions, which might not always be respected in practical scenarios. For example, VarLiNGAM assumes non-Gaussian errors, which is not the case in our experiments. Similarly, we use PCMCI with the ParCorr independence test; while a nonlinear test would be more appropriate. We faced computational challenges with PCMCI when attempting to use its nonlinear inpendence test CMIknn. Nevertheless, our goal is not to demonstrate that our approach outperforms all others under all conditions. Instead, we aim to show that our method can be a valuable addition to the toolbox for causal discovery in time series, offering unique insights and potentially complementing existing techniques. The adopted conditions for each method might be suboptimal for the given dataset, yet they provide a robust benchmark to evaluate the relative strengths and potential applications of our proposed approach.

Each method has been wrapped conveniently to uniformize the way to create the objects, run the execution, and return the same structure.  

In [1]:
import os

# This is because VARLINGAM will use all available CPU with n_jobs > 1 - Limit to 1 thread
os.environ['MKL_NUM_THREADS'] = '1'  
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.append('../src')


import pickle 
import os
from d2c.descriptors import DataLoader
from d2c.benchmark import VARLiNGAM, PCMCI, Granger, DYNOTEARS, D2CWrapper, VAR, MultivariateGranger

from imblearn.ensemble import BalancedRandomForestClassifier

#suppress future warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [2]:
N_VARS = 5
N_JOBS = 50
MAXLAGS = 3

## Explanation
We start by loading our test data and collecting the corresponding observations. <br>
Notice that we collect the original observations rather than the lagged one. <br>
This is because the methods will build lag matrices internally. <br>

At this stage it's important to introduce the concept of `causal_df`. <br> 
A `causal_df` is a representation of the output of causal discovery.<br> 
It's a dataframe containing the following columns: <br> 
`['from', 'to', 'effect', 'p_value', 'probability', 'is_causal']`

- `from`: the source variable (in the past)
- `to`: the target variable (in the present)
- `effect`: the estimated effect size (if provided by the method)
- `p_value`: the p-value associated with the effect (if provided by the method)
- `probability`: the probability of the causal relationship (if provided by the method)
- `is_causal`: a boolean indicating whether the relationship is causal <br>

We remind our variable naming convention: <br>
- A time series of `n_variables` dimensions will have names from 0 to `n_variables - 1` to refer to the variables at time `t` (present)
- names from `n_variables` to `n_variables*2 - 1` will indicate the same variable at time `t-1` (1-lag)
- names from `n_variables*2` to `n_variables*3 - 1` will indicate the same variable at time `t-2` (2-lag)  <br>
For example if `n_variables = 5`, the line where `from` is 12 and `to` is 4, refers to the link between variable `3` at `t-2` to variable `5` at time `t` <br>

Here is an example of a `causal_df` for 5 variables

|   |    from | to  |  effect  | p_value |probability |is_causal |
|------|---------|-----|----------|---------|------------|----------|
|     |   5  | 0 |  0.52155 | 0.313978   |     None    |     0     |
|     |   5  | 1 |-0.006598 | 0.059683   |     None    |     0     |
|     |   5  | 2 | 0.445405 | 0.968117   |     None    |     0     |
|     |   5  | 3 | 0.017567 | 0.033022   |     None    |     0     |
|     |   5  | 4 | -0.04921 | 0.205457   |     None    |     0     |
|    | ...  |.. |      ... |      ...   |      ...    |   ...     |
|   |  29  | 0 | 0.064319 | 0.170674   |     None    |     0     |
|   |  29  | 1 |-0.059419 | 0.550454   |     None    |     0     |
|   |  29  | 2 |-0.017221 | 0.829726   |     None    |     0     |
|   |  29  | 3 | 0.020435 | 0.218739   |     None    |     0     |
|   |  29  | 4 | 0.042379 | 0.294798   |     None    |     0     |

In [11]:
dataloaders = {}
original_observations_testing = {} 
lagged_flattened_observations_testing = {} 
flattened_dags_testing = {}
true_causal_dfs = {}

for error_dist in ['gaussian', 'uniform', 'laplace']:
    dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
    dataloader.from_pickle(f'data/testing_data_{error_dist}.pkl')
    
    dataloaders[error_dist] = dataloader
    original_observations_testing[error_dist] = dataloader.get_original_observations()
    lagged_flattened_observations_testing[error_dist] = dataloader.get_observations()
    flattened_dags_testing[error_dist] = dataloader.get_dags()
    true_causal_dfs[error_dist] = dataloader.get_true_causal_dfs()

original_observations_list_testing = []
for obs_list in original_observations_testing.values():
    original_observations_list_testing.extend(obs_list) 

lagged_flattened_observations_list_testing = []
for obs_list in lagged_flattened_observations_testing.values():
    lagged_flattened_observations_list_testing.extend(obs_list)

flattened_dags_list_testing = []
for dags_list in flattened_dags_testing.values():
    flattened_dags_list_testing.extend(dags_list)

true_causal_dfs_list_testing = []
for causal_df in true_causal_dfs.values():
    true_causal_dfs_list_testing.extend(causal_df)

Each method from the benchmark will take as input 
- `ts_list`: a list of `np.arrays` containing the values of the time series, 
- `maxlags`: the maxlags, 
- `n_jobs`: the number of jobs. <br>
It's important to notice that `get_causal_dfs()` will return a dictionary of dataframes where the key is the index of the corresponding time series from the input list `ts_list`.
So, if our data contains `15` time series and you want to access the last one we can do `causal_dfs[15 - 1]`

## Competitors

In [16]:
var = VAR(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
var.run()
causal_dfs_var = var.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [17]:
varlingam = VARLiNGAM(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
varlingam.run()
causal_dfs_varlingam = varlingam.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [18]:
pcmci = PCMCI(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
pcmci.run()
causal_dfs_pcmci = pcmci.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [19]:
granger = Granger(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
granger.run()
causal_dfs_granger = granger.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [20]:
dynotears = DYNOTEARS(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
dynotears.run()
causal_dfs_dynotears = dynotears.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [21]:
mvgc = MultivariateGranger(ts_list=original_observations_list_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
mvgc.run()
causal_dfs_mvgc = mvgc.get_causal_dfs()

Running parallel inference over 1080 time series using 60 jobs...


Processing Time Series:   0%|          | 0/1080 [00:00<?, ?it/s]

In [ ]:
# save all in a temp folder pickle 
temp_folder = 'data/benchmark_results'
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)


In [21]:
causal_dfs_pcmci_gpdc = pickle.load(open(os.path.join(temp_folder, 'causal_dfs_pcmci_gdgc.pkl'), 'rb'))

In [6]:

all = {
    'causal_dfs_var': causal_dfs_var,
    'causal_dfs_varlingam': causal_dfs_varlingam,
    'causal_dfs_pcmci': causal_dfs_pcmci,
    # 'causal_dfs_pcmci_gpdc': causal_dfs_pcmci_gpdc, 
    'causal_dfs_granger': causal_dfs_granger,
    'causal_dfs_mvgc': causal_dfs_mvgc,
    'causal_dfs_dynotears': causal_dfs_dynotears,
    'observations': original_observations_list_testing,
    'dags': flattened_dags_list_testing,
    'true_causal_dfs': true_causal_dfs_list_testing
}
pickle.dump(all, open(os.path.join(temp_folder, 'causal_dfs_before_d2c.pkl'), 'wb'))

NameError: name 'causal_dfs_var' is not defined

In [22]:
# load everything
all = pickle.load(open(os.path.join(temp_folder, 'causal_dfs_before_d2c.pkl'), 'rb'))
causal_dfs_var = all['causal_dfs_var']
causal_dfs_varlingam = all['causal_dfs_varlingam']
causal_dfs_pcmci = all['causal_dfs_pcmci']
causal_dfs_granger = all['causal_dfs_granger']
causal_dfs_dynotears = all['causal_dfs_dynotears']

## D2CWrapper
For coherence with the other results, a D2CWrapper class has been created that behave exactly like the other approaches.<br>
It therefore exposes the methods `run()` and `get_causal_dfs()`. <br>
It requires a model that has been trained already and it will compute descriptors for unseen data of which the DAG is ignored. <br>
In this case, the model cannot select a subset of features (no `couples_to_consider_per_dag` attribute).
The predictions from the model on the newly computed descriptors are the labels that will be provided in the causal df. <br>

<b>Important:</b> make sure your model has been trained on the same feature set. If you have used `full=True` when generating the training descriptors, you should use `full=True` here as well

In [5]:
import pandas as pd
descriptors_df_train = pd.read_pickle('data/descriptors_df_train.pkl')

X_train = descriptors_df_train.drop(columns=['graph_id','edge_source','edge_dest','is_causal'])
y_train = descriptors_df_train['is_causal']

clf = BalancedRandomForestClassifier(n_estimators=500, max_depth=None, random_state=0, sampling_strategy='auto',replacement=True,bootstrap=True)
clf.fit(X_train, y_train)

BalancedRandomForestClassifier(bootstrap=True, n_estimators=500, random_state=0,
                               replacement=True, sampling_strategy='auto')

In [6]:
len(lagged_flattened_observations_list_testing)

1080

In [12]:
N_JOBS

50

In [13]:
# sample from list of lagged_flattened_observations_list_testing
# and sample in the same way true_causal_dfs
# randomly N_JOBS samples
import random
indeces = random.sample(range(len(lagged_flattened_observations_list_testing)), N_JOBS)
lagged_flattened_observations_testing_sampled = [lagged_flattened_observations_list_testing[i] for i in indeces]
flattened_dags_testing_sampled = [flattened_dags_list_testing[i] for i in indeces]
true_causal_dfs_testing_sampled = [true_causal_dfs_list_testing[i] for i in indeces]

In [15]:
from d2c.descriptors import D2C, DataLoader
d2c_new = D2C(observations=lagged_flattened_observations_testing_sampled,
        dags=flattened_dags_testing_sampled, 
        couples_to_consider_per_dag=-1, 
        n_variables=N_VARS, 
        maxlags=MAXLAGS,
        seed=42,
        n_jobs=N_JOBS,
        full=True,
        dynamic=True,
        mb_estimator='ts',
        )

d2c_new.initialize()
d2c_new.get_descriptors_df().to_pickle('data/random_subset_50_descriptors_test.pkl')

Processing DAGs:   0%|          | 0/50 [00:00<?, ?it/s]

In [8]:
d2cwrapper = D2CWrapper(
    ts_list=sampled_lagged_flattened_observations_list_testing,
    model=clf,
    n_variables=N_VARS,
    maxlags=MAXLAGS,
    mb_estimator = 'ts',
    n_jobs=N_JOBS, 
    full=True,
    dynamic=True,
    manages_own_parallelism=False,
    filename='descriptors_test/descriptors'
)


d2cwrapper.run()
causal_dfs_d2c = d2cwrapper.get_causal_dfs()

Running parallel inference over 50 time series using 50 jobs...


Processing Time Series:   0%|          | 0/50 [00:00<?, ?it/s]

Saving descriptors to descriptors_test/descriptors_descriptors_5.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_6.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_14.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_22.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_49.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_18.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_17.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_15.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_13.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_7.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_32.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_12.csv...
Saving descriptors to descriptors_test/descriptors_descriptors_25.csv...
Saving descriptors to descriptors_test/descriptors_des

In [10]:
# merge all csv in descriptors_test folder 
import os 
import pandas as pd
descriptors_test_folder = 'descriptors_test'
all_descriptors = []
for filename in os.listdir(descriptors_test_folder):
    if filename.endswith('.csv'):
        df = pd.read_csv(os.path.join(descriptors_test_folder, filename))
        all_descriptors.append(df)

descriptors_df = pd.concat(all_descriptors, ignore_index=True)
# save descriptors_df to pickle
descriptors_df.to_pickle('data/random_subset_50_descriptors_test.pkl')

## Saving

In [15]:
with open('data/causal_dfs.pkl', 'rb') as f:
    causal_dfs_var, causal_dfs_varlingam, causal_dfs_pcmci, causal_dfs_granger, causal_dfs_dynotears, causal_dfs_d2c, true_causal_dfs = pickle.load(f)
    

EOFError: Ran out of input

In [ ]:
with open('data/causal_dfs_proxy.pkl', 'wb') as f:
    pickle.dump((causal_dfs_var, 
                causal_dfs_varlingam, 
                causal_dfs_pcmci,
                causal_dfs_mvgc,
                # causal_dfs_pcmci_gpdc,
                causal_dfs_granger, 
                causal_dfs_dynotears,
                causal_dfs_d2c, 
                true_causal_dfs), f)